##### **Aplicación de indicadores de temática asistida por LLMs**

Entrenamos 3 modelos *Zero-Shot Learning* diseñados con conocimiento "general" y que, aplicados a nuevos ejemplos, nos ayudarán a identificar la temática de cada titular entre unas categorías dadas. 

Hacemos en primer lugar un análisis de los titulares de nuestro conjunto de datos para extraer las temáticas predominantes que aparecen en él y poder identificar las categorías que pasaremos a los modelos *Zero-Shot*. 

Realizamos el análisis en el conjunto obtenido de haber procesado los datos para la extracción de tópicos previamente por LDA dado que nos facilita el reconocimiento de temáticas.

In [35]:
# Cargamos el conjunto de datos 
import pandas as pd

df = pd.read_csv("dataset_universidades_preprocesado_topics.csv")

In [32]:
# Importación de librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.cluster import KMeans
import itertools
from transformers import pipeline
import ast

In [36]:
## ANALISIS DE FRECUENCIAS DE TÓPICOS Y TOKENS DE CARA A DEFINIR LOS TÓPICOS POR LOS QUE SE PREGUNTARÁ A LOS MODELOS ZERO-SHOT


# 1. ANÁLISIS DE TÉRMINOS MÁS FRECUENTES
# -----------------------------------------------------------
print("=== ANÁLISIS DE TÉRMINOS MÁS FRECUENTES ===")

df['processed_tokens'] = df['processed_tokens'].apply(ast.literal_eval)

# Calcular frecuencias de todos los tokens
all_tokens = list(itertools.chain.from_iterable(df['processed_tokens']))
token_freq = Counter(all_tokens)

# Top 100 términos más frecuentes
top_100_terms = token_freq.most_common(100)
print(f"\nTop 50 términos más frecuentes (de {len(token_freq)} términos únicos):")
for i, (term, freq) in enumerate(top_100_terms[:50], 1):
    print(f"{i:2d}. {term:20s} - {freq:6d} apariciones")

# 2. ANÁLISIS DE BIGRAMAS (PAREJAS DE PALABRAS)
# -----------------------------------------------------------
print("\n=== ANÁLISIS DE BIGRAMAS ===")

# Generar bigramas
bigrams = []
for tokens in df['processed_tokens']:
    if len(tokens) >= 2:
        bigrams.extend([f"{tokens[i]} {tokens[i+1]}" for i in range(len(tokens)-1)])

bigram_freq = Counter(bigrams)
top_50_bigrams = bigram_freq.most_common(50)

print("Top 30 bigramas más frecuentes:")
for i, (bigram, freq) in enumerate(top_50_bigrams[:30], 1):
    print(f"{i:2d}. {bigram:30s} - {freq:6d} apariciones")

# 3. ANÁLISIS TF-IDF PARA TÉRMINOS MÁS ESPECÍFICOS
# -----------------------------------------------------------
print("\n=== ANÁLISIS TF-IDF (TÉRMINOS ESPECÍFICOS) ===")

# Convertir a texto para TF-IDF
documents = [' '.join(tokens) for tokens in df['processed_tokens']]

tfidf_vectorizer = TfidfVectorizer(
    max_features=1000,
    min_df=10,
    max_df=0.4,
    stop_words=[]
)

tfidf_matrix = tfidf_vectorizer.fit_transform(documents)
feature_names = tfidf_vectorizer.get_feature_names_out()

# Promedio de TF-IDF por término
mean_tfidf = np.array(tfidf_matrix.mean(axis=0)).flatten()

# Top términos por TF-IDF (más específicos)
tfidf_ranking = sorted(zip(feature_names, mean_tfidf),
                       key=lambda x: x[1], reverse=True)

print("Top 30 términos por TF-IDF (más específicos):")
for i, (term, score) in enumerate(tfidf_ranking[:30], 1):
    print(f"{i:2d}. {term:20s} - {score:.4f}")


# 7. ANÁLISIS DE TENDENCIAS TEMPORALES DE TÉRMINOS
# -----------------------------------------------------------
print("\n=== TENDENCIAS TEMPORALES DE TÉRMINOS PRINCIPALES ===")

# Analizar evolución de términos clave
key_terms = [term for term, freq in top_100_terms[:20]]

# Agrupar por quinquenio y contar términos
df['text_str'] = df['processed_tokens'].apply(lambda x: ' '.join(x))
term_trends = {}

for term in key_terms[:10]:  # Solo primeros 10 para no saturar
    trend = []
    for quinq in sorted(df['quinquenio'].unique()):
        mask = df['quinquenio'] == quinq
        docs_in_quinq = df[mask]['text_str']
        count = docs_in_quinq.str.contains(term, regex=False).sum()
        total_docs = mask.sum()
        trend.append((count / total_docs) * 100 if total_docs > 0 else 0)
    term_trends[term] = trend

# Mostrar tendencias
trend_df = pd.DataFrame(term_trends, index=sorted(df['quinquenio'].unique()))
print("\nEvolución porcentual de términos clave por quinquenio:")
print(trend_df.round(2))

# 8. RESUMEN PARA ZERO-SHOT
# -----------------------------------------------------------
print("\n" + "="*60)
print("RESUMEN PARA SELECCIÓN DE TEMÁTICAS ZERO-SHOT")
print("="*60)

print("\nTEMÁTICAS SUGERIDAS BASADAS EN EL ANÁLISIS:")

# Sugerencias basadas en términos frecuentes
thematic_suggestions = {
    'educación': ['educación', 'universidad', 'estudios', 'alumnos', 'docentes'],
    'salud': ['sanitario', 'salud', 'médico', 'hospital', 'enfermedad'],
    'economía': ['económico', 'presupuesto', 'financiero', 'empresa', 'mercado'],
    'empleo': ['trabajo', 'empleo', 'laboral', 'contrato', 'desempleo'],
    'administración': ['administrativo', 'procedimiento', 'registro', 'oficina', 'expediente'],
    'medio_ambiente': ['ambiental', 'medio', 'contaminación', 'naturaleza', 'sostenible'],
    'tecnología': ['tecnológico', 'digital', 'informático', 'internet', 'electrónico'],
    'transporte': ['transporte', 'vehículo', 'tráfico', 'carretera', 'movilidad'],
    'vivienda': ['vivienda', 'urbanístico', 'construcción', 'inmobiliario', 'alquiler'],
    'seguridad': ['seguridad', 'policial', 'delito', 'protección', 'emergencia']
}

# Verificar qué temáticas están presentes
print("\nTemáticas detectadas en el corpus:")
for theme, keywords in thematic_suggestions.items():
    matches = [kw for kw in keywords if kw in token_freq]
    if len(matches) >= 3:  # Al menos 3 palabras clave presentes
        print(f"{theme.upper()}: {matches}")

=== ANÁLISIS DE TÉRMINOS MÁS FRECUENTES ===

Top 50 términos más frecuentes (de 27971 términos únicos):
 1. extravío             -  41110 apariciones
 2. concurso             -  27465 apariciones
 3. madrid               -  23649 apariciones
 4. público              -  22785 apariciones
 5. conocimiento         -  21620 apariciones
 6. convocar             -  20407 apariciones
 7. estudio              -  18873 apariciones
 8. plan                 -  17853 apariciones
 9. educación            -  12098 apariciones
10. ciencia              -  12091 apariciones
11. politécnico          -  10709 apariciones
12. complutense          -  10326 apariciones
13. anunciar             -   8786 apariciones
14. departamento         -   8256 apariciones
15. valencia             -   8064 apariciones
16. autónomo             -   7683 apariciones
17. granada              -   7644 apariciones
18. contratación         -   7626 apariciones
19. profesoro            -   7624 apariciones
20. ingeniería        

---

Una vez analizadas las temáticas presentes en nuestro conjunto de datos y decididas las categorías que les pasaremos a nuestros modelos *Zero-Shot*, creamos un dataset con algunos de los titulares que tenemos y tres columnas vacías (una por cada modelo que vamos a utilizar), de cara a guardar los resultados de los diferentes modelos y poder compararlos para elegir el consenso de clasificación.

Vamos a realizar primero pequeñas pruebas de diferentes modelos para ver cúan parecidos son los resultados, y para ello cogemos 20 filas aleatorias de cada quinquenio.

In [17]:
# Crear el nuevo dataset con 20 filas aleatorias por cada quinquenio
df_modelos_zeroshot = pd.DataFrame()

# Obtener los valores únicos de la columna "quinquenio"
quinquenios = df['quinquenio'].unique()

# Para cada quinquenio, seleccionar 20 filas aleatorias
for quinquenio in quinquenios:
    # Filtrar por quinquenio y seleccionar 20 filas aleatorias
    muestra_quinquenio = df[df['quinquenio'] == quinquenio].sample(n=20, random_state=42)

    # Crear el DataFrame temporal para este quinquenio
    df_temp = pd.DataFrame({
        'titulo': muestra_quinquenio['Titulo_Semantico'],
        'modelo1': '',
        'modelo2': '',
        'modelo3': '',
        'quinquenio': quinquenio  # Mantener la columna quinquenio para referencia
    })

    # Concatenar al DataFrame principal
    df_modelos_zeroshot = pd.concat([df_modelos_zeroshot, df_temp], ignore_index=True)

print("Nuevo dataset creado exitosamente!")
print(f"Dimensiones del nuevo dataset: {df_modelos_zeroshot.shape}")
print(f"Número de quinquenios únicos: {len(quinquenios)}")
print(f"Total de filas esperadas: {len(quinquenios) * 20}")
print("\nPrimeras 5 filas del nuevo dataset:")
print(df_modelos_zeroshot.head())

Nuevo dataset creado exitosamente!
Dimensiones del nuevo dataset: (120, 5)
Número de quinquenios únicos: 6
Total de filas esperadas: 120

Primeras 5 filas del nuevo dataset:
                                              titulo modelo1 modelo2 modelo3  \
0  Resolución de 15 de marzo de 1999, de la Unive...                           
1  Resolución de 21 de febrero de 1996, de la Uni...                           
2  Resolución de 29 de octubre de 1997, de la Uni...                           
3  Resolución de 23 de febrero de 1998, de la Uni...                           
4  Resolución de 8 de septiembre de 1999, de la U...                           

   quinquenio  
0        1995  
1        1995  
2        1995  
3        1995  
4        1995  


---

#### **ZERO-SHOT LEANING**

Estos modelos pueden clasificar texto en categorías que no han visto durante el entrenamiento, usando comprensión semántica y comparación entre el texto de entrada y las descripciones de las categorías.

1. **MoritzLaurer/deberta-v3-base-zeroshot-v1.1-all-33**: basado en el modelo DeBERTa pero entrenado específicamente para tareas de clasificación múltiple (fine-tunning) comparando directamente el texto con las posibles clases que le proporciones. Está además entrenado de forma multilingüe (aunque es más fuerte en inglés) dado que ha visto ejemplos en varios idiomas durante su especialización, no solo durante el pre-entrenamiento base.

In [ ]:
## PRUEBA EN TITULO INDIVIDUAL
#!pip install transformers[sentencepiece]
import torch

text = "Resolución de 30 de noviembre de 1994, de la Universidad Politécnica de Madrid, por la que se convoca a libre designación, entre funcionarios de carrera, puesto vacante en esta Universidad."
classes_verbalized = ["salud", "economía", "empleo", "tecnología"]
zeroshot_classifier = pipeline("zero-shot-classification", model="MoritzLaurer/deberta-v3-base-zeroshot-v1.1-all-33")
output = zeroshot_classifier(text, classes_verbalized, multi_label=True)
print(output)

Device set to use cuda:0
{'sequence': 'Resolución de 30 de noviembre de 1994, de la Universidad Politécnica de Madrid, por la que se convoca a libre designación, entre funcionarios de carrera, puesto vacante en esta Universidad.', 'labels': ['empleo', 'tecnología', 'economía', 'salud'], 'scores': [0.865053117275238, 0.08733005076646805, 0.00039666122756898403, 5.3491330618271604e-05]}


In [ ]:
#!pip install transformers[sentencepiece] torch
import torch

# EJECUCIÓN EN NUESTRO SUBCONJUNTO DE DATOS COMPLETO

# Configuración de ejecución
device = 0 if torch.cuda.is_available() else -1
if torch.cuda.is_available():
    print(f"GPU detectada: {torch.cuda.get_device_name(0)}")
else:
    print(" No hay GPU disponible")

# Inicializar el clasificador zero-shot
classes_verbalized = ["salud", "economía", "empleo", "tecnología"]
zeroshot_classifier = pipeline("zero-shot-classification", model="MoritzLaurer/deberta-v3-base-zeroshot-v1.1-all-33")

# Lista para almacenar los resultados
resultados_modelo1 = []

print("Analizando títulos con el modelo...")

# Procesar cada título
for i, titulo in enumerate(df_modelos_zeroshot['titulo']):
    try:
        # Clasificar el texto
        output = zeroshot_classifier(titulo, classes_verbalized, multi_label=True)

        # Crear diccionario con las clases y sus scores
        resultado_dict = {}
        for label, score in zip(output['labels'], output['scores']):
            resultado_dict[label] = float(score)

        # Guardar el diccionario como string
        resultados_modelo1.append(str(resultado_dict))

        if (i + 1) % 20 == 0:  # Mostrar progreso cada 10 títulos
            print(f"Procesados {i + 1}/{len(df_modelos_zeroshot)} títulos")

    except Exception as e:
        print(f"Error procesando título {i + 1}: {e}")
        resultados_modelo1.append("{}")  # Diccionario vacío en caso de error

# Asignar los resultados a la columna modelo1
df_modelos_zeroshot['modelo1'] = resultados_modelo1

print(f"\nAnálisis completado!")
print(f"Total de títulos procesados: {len(df_modelos_zeroshot)}")

# Mostrar algunos ejemplos
print("\nPrimeros 3 resultados:")
for i in range(min(3, len(df_modelos_zeroshot))):
    print(f"\nTítulo {i+1}: {df_modelos_zeroshot['titulo'].iloc[i][:100]}...")
    print(f"Modelo1: {df_modelos_zeroshot['modelo1'].iloc[i]}")

GPU detectada: Tesla T4
Device set to use cuda:0
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Analizando títulos con el modelo...
Procesados 20/120 títulos
Procesados 40/120 títulos
Procesados 60/120 títulos
Procesados 80/120 títulos
Procesados 100/120 títulos
Procesados 120/120 títulos

Análisis completado!
Total de títulos procesados: 120

Primeros 3 resultados:

Título 1: Resolución de 15 de marzo de 1999, de la Universidad Complutense de Madrid, por la que se nombra a d...
Modelo1: {'empleo': 0.0031115086749196053, 'economía': 0.0002642683102749288, 'tecnología': 0.00023413581948261708, 'salud': 8.51429722388275e-05}

Título 2: Resolución de 21 de febrero de 1996, de la Universidad Politécnica de Madrid, por la que se corrige ...
Modelo1: {'tecnología': 0.11225901544094086, 'empleo': 0.0064381021074950695, 'economía': 0.0051696584559977055, 'salud': 7.141901005525142e-05}

Título 3: Reso

2. **MoritzLaurer/deberta-v3-large-zeroshot-v1**: modelo muy similar al anterior también basado en DeBERTa y con alto rendimiento en clasificación multi-clase (trabaja con múltiples etiquetas simultáneamente), bajo la que asigna una probabilidad a cada clase y elige la más adecuada incluso si son conceptualmente similares.  Está optimizado además para baja latencia equilibrando precisión y eficiencia computacional y entrenado para comprender mejor relaciones sintácticas y estructuras gramaticales complejas.

In [ ]:
# PRUEBA DEL MODELO EN TEXTO INDIVIDUAL

text = "Resolución de 30 de noviembre de 1994, de la Universidad Politécnica de Madrid, por la que se convoca a libre designación, entre funcionarios de carrera, puesto vacante en esta Universidad."
# hypothesis_template = "This example is about {}"
classes_verbalized = ["salud", "economía", "empleo", "tecnología"]
zeroshot_classifier = pipeline("zero-shot-classification", model="MoritzLaurer/deberta-v3-large-zeroshot-v1")
output = zeroshot_classifier(text, classes_verbalized, multi_label=True)
print(output)

Device set to use cuda:0
{'sequence': 'Resolución de 30 de noviembre de 1994, de la Universidad Politécnica de Madrid, por la que se convoca a libre designación, entre funcionarios de carrera, puesto vacante en esta Universidad.', 'labels': ['empleo', 'tecnología', 'economía', 'salud'], 'scores': [0.9959377646446228, 0.055157773196697235, 3.282725083408877e-05, 1.532610804133583e-05]}


In [ ]:
# CONFIGURACION
device = 0 if torch.cuda.is_available() else -1
if torch.cuda.is_available():
    print(f" GPU detectada: {torch.cuda.get_device_name(0)}")
else:
    print(" No hay GPU disponible")

# Inicializar el clasificador zero-shot
classes_verbalized = ["salud", "economía", "empleo", "tecnología"]
zeroshot_classifier = pipeline("zero-shot-classification", model="MoritzLaurer/deberta-v3-large-zeroshot-v1")

# Lista para almacenar los resultados
resultados_modelo2 = []

print("Analizando títulos con el modelo...")

# Procesar cada título
for i, titulo in enumerate(df_modelos_zeroshot['titulo']):
    try:
        # Clasificar el texto
        output = zeroshot_classifier(titulo, classes_verbalized, multi_label=True)

        # Crear diccionario con las clases y sus scores
        resultado_dict = {}
        for label, score in zip(output['labels'], output['scores']):
            resultado_dict[label] = float(score)

        # Guardar el diccionario como string
        resultados_modelo2.append(str(resultado_dict))

        if (i + 1) % 20 == 0:  # Mostrar progreso cada 10 títulos
            print(f"Procesados {i + 1}/{len(df_modelos_zeroshot)} títulos")

    except Exception as e:
        print(f"Error procesando título {i + 1}: {e}")
        resultados_modelo2.append("{}")  # Diccionario vacío en caso de error

# Asignar los resultados a la columna modelo1
df_modelos_zeroshot['modelo2'] = resultados_modelo2

print(f"\nAnálisis completado!")
print(f"Total de títulos procesados: {len(df_modelos_zeroshot)}")

# Mostrar algunos ejemplos
print("\nPrimeros 3 resultados:")
for i in range(min(3, len(df_modelos_zeroshot))):
    print(f"\nTítulo {i+1}: {df_modelos_zeroshot['titulo'].iloc[i][:100]}...")
    print(f"Modelo2: {df_modelos_zeroshot['modelo2'].iloc[i]}")

GPU detectada: Tesla T4
Device set to use cuda:0
Analizando títulos con el modelo...
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Procesados 20/120 títulos
Procesados 40/120 títulos
Procesados 60/120 títulos
Procesados 80/120 títulos
Procesados 100/120 títulos
Procesados 120/120 títulos

Análisis completado!
Total de títulos procesados: 120

Primeros 3 resultados:

Título 1: Resolución de 15 de marzo de 1999, de la Universidad Complutense de Madrid, por la que se nombra a d...
Modelo1: {'empleo': 0.029507363215088844, 'tecnología': 1.606160913070198e-05, 'economía': 1.3315580872585997e-05, 'salud': 1.3058037438895553e-05}

Título 2: Resolución de 21 de febrero de 1996, de la Universidad Politécnica de Madrid, por la que se corrige ...
Modelo1: {'tecnología': 0.028925716876983643, 'empleo': 7.003774226177484e-05, 'economía': 5.3910855058347806e-05, 'salud': 1.4567269317922182e-05}

Título 3: Resolución de 29 de octubre de 1

3. **Recognai/zeroshot_selectra_medium**: modelo específicamente diseñado para el español y que interpreta las categorías basándose en su conocimiento del idioma. Basado en un modelo Selectra pre-entrenado y especificado en tareas de clasificación para aprender a relacionar textos en español con diferentes categorías; su versión “Medium” ofrece un buen equilibrio entre rendimiento y eficiencia computacional siendo más rápido pero manteniendo buena precisión. Entiende particularidades del español como modismos, dobles sentidos, y registros formales/informales típicos del idioma.

In [ ]:
# PRUEBA DEL MODELO EN TEXTO INDIVIDUAL
text = "Resolución de 30 de noviembre de 1994, de la Universidad Politécnica de Madrid, por la que se convoca a libre designación, entre funcionarios de carrera, puesto vacante en esta Universidad."
classes_verbalized = ["salud", "economía", "empleo", "tecnología"]
zeroshot_classifier = pipeline("zero-shot-classification", model="Recognai/zeroshot_selectra_medium")
output = zeroshot_classifier(text, classes_verbalized, multi_label=True)
print(output)

Device set to use cuda:0
{'sequence': 'Resolución de 30 de noviembre de 1994, de la Universidad Politécnica de Madrid, por la que se convoca a libre designación, entre funcionarios de carrera, puesto vacante en esta Universidad.', 'labels': ['empleo', 'tecnología', 'economía', 'salud'], 'scores': [0.865053117275238, 0.08733005076646805, 0.00039666122756898403, 5.3491330618271604e-05]}


In [ ]:
# CONFIGURACION
device = 0 if torch.cuda.is_available() else -1
if torch.cuda.is_available():
    print(f" GPU detectada: {torch.cuda.get_device_name(0)}")
else:
    print(" No hay GPU disponible")

# Inicializar el clasificador zero-shot
classes_verbalized = ["salud", "economía", "empleo", "tecnología"]
zeroshot_classifier = pipeline("zero-shot-classification", model="Recognai/zeroshot_selectra_medium")

# Lista para almacenar los resultados
resultados_modelo3 = []

print("Analizando títulos con el modelo...")

# Procesar cada título
for i, titulo in enumerate(df_modelos_zeroshot['titulo']):
    try:
        # Clasificar el texto
        output = zeroshot_classifier(titulo, classes_verbalized, multi_label=True)

        # Crear diccionario con las clases y sus scores
        resultado_dict = {}
        for label, score in zip(output['labels'], output['scores']):
            resultado_dict[label] = float(score)

        # Guardar el diccionario como string
        resultados_modelo3.append(str(resultado_dict))

        if (i + 1) % 20 == 0:  # Mostrar progreso cada 10 títulos
            print(f"Procesados {i + 1}/{len(df_modelos_zeroshot)} títulos")

    except Exception as e:
        print(f"Error procesando título {i + 1}: {e}")
        resultados_modelo3.append("{}")  # Diccionario vacío en caso de error

# Asignar los resultados a la columna modelo1
df_modelos_zeroshot['modelo3'] = resultados_modelo3

print(f"\nAnálisis completado!")
print(f"Total de títulos procesados: {len(df_modelos_zeroshot)}")

# Mostrar algunos ejemplos
print("\nPrimeros 3 resultados:")
for i in range(min(3, len(df_modelos_zeroshot))):
    print(f"\nTítulo {i+1}: {df_modelos_zeroshot['titulo'].iloc[i][:100]}...")
    print(f"Modelo3: {df_modelos_zeroshot['modelo3'].iloc[i]}")

GPU detectada: Tesla T4
Device set to use cuda:0
Analizando títulos con el modelo...
Procesados 20/120 títulos
Procesados 40/120 títulos
Procesados 60/120 títulos
Procesados 80/120 títulos
Procesados 100/120 títulos
Procesados 120/120 títulos

Análisis completado!
Total de títulos procesados: 120

Primeros 3 resultados:

Título 1: Resolución de 15 de marzo de 1999, de la Universidad Complutense de Madrid, por la que se nombra a d...
Modelo1: {'empleo': 0.0585443377494812, 'tecnología': 0.0010448237881064415, 'salud': 0.000986192375421524, 'economía': 0.0003174387093167752}

Título 2: Resolución de 21 de febrero de 1996, de la Universidad Politécnica de Madrid, por la que se corrige ...
Modelo1: {'empleo': 0.011649413034319878, 'tecnología': 0.007281607482582331, 'economía': 0.002663959749042988, 'salud': 0.0019535720348358154}

Título 3: Resolución de 29 de octubre de 1997, de la Universidad de Lleida, por la que se publica la modificac...
Modelo1: {'empleo': 0.019020594656467438, 'sal

In [ ]:
# Imprimimos por pantalla el resultado final para ver una pequeña traza del funcionamiento de los modelos
df_modelos_zeroshot

---

Usando un mecanismo de “consenso basado en triple medición” decidimos que parece funcionar mejor el segundo modelo puesto que siempre está “de acuerdo” con alguno de los otros dos (clasifican la misma temática/categoría). Decidimos procesar el conjunto entero de datos con dicho segundo modelo (“MoritzLaurer/deberta-v3-large-zeroshot-v1”) de cara a obtener un valor asociado a cada categoría para cada titular. 

Dividimos para ello el conjunto en 3 partes iguales dada nuestra capacidad computacional algo limitada:

In [ ]:
def dividir_datos_3_partes(data):
    """
    Divide un DataFrame en 3 partes iguales

    Args:
        data: DataFrame de pandas

    Returns:
        tuple: Tres DataFrames con el mismo número de filas
    """
    n = len(data)
    tamano_parte = n // 3

    # Calcular los índices de división
    idx1 = tamano_parte
    idx2 = 2 * tamano_parte

    # Dividir el DataFrame
    parte1 = data.iloc[:idx1]
    parte2 = data.iloc[idx1:idx2]
    parte3 = data.iloc[idx2:]

    return parte1, parte2, parte3


# Dividir los datos
data1, data2, data3 = dividir_datos_3_partes(df)

print(f"Parte 1: {len(data1)} filas")
print(f"Parte 2: {len(data2)} filas")
print(f"Parte 3: {len(data3)} filas")

Parte 1: 54191 filas
Parte 2: 54191 filas
Parte 3: 54192 filas


In [ ]:
# Guardamos los subconjuntos creados
data1.to_csv("df_primer_tercio.csv", index=False)
data2.to_csv("df_segundo_tercio.csv", index=False)
data3.to_csv("df_tercer_tercio.csv", index=False)

---

Con ello entonces ejecutamos el modelo por partes y vamos guardando los resultados para posteriormente juntar todo en el conjunto original y combinarlo con el resto de indicadores que se han realizado en el proyecto.

Se expone la ejecución del segundo tercio del conjunto de datos.

In [ ]:
# Posible instalación necesaria para evitar fallos en versiones
# !pip install -U transformers==4.44.2 huggingface_hub==0.24.6 

In [ ]:
# Comprobar antes de ejecución y sino ejecutar la instalación
import transformers
import huggingface_hub
print(transformers.__version__, huggingface_hub.__version__)


4.44.2 0.24.6


In [ ]:
# ------------------------------------------------------------
# CONFIGURACIÓN
# ------------------------------------------------------------
device = 0 if torch.cuda.is_available() else -1
print(f"Usando dispositivo: {'GPU' if device == 0 else 'CPU'}")

# Etiquetas de clasificación
classes_verbalized = ["salud", "economía", "empleo", "tecnología"]

# Cargar modelo zero-shot
model_name = "MoritzLaurer/deberta-v3-large-zeroshot-v1"
zeroshot_classifier = pipeline(
    "zero-shot-classification",
    model=model_name,
    device=device,
    trust_remote_code=False
)


# ------------------------------------------------------------
# PROCESAMIENTO EN GPU CON BATCHING INTERNO
# ------------------------------------------------------------
print("Analizando títulos en GPU con batching interno...")

# Ajusta el tamaño del batch según la VRAM de tu GPU
batch_size = 64

# Ejecutar inferencia por lotes directamente
results = zeroshot_classifier(
    df["texto_lower"].tolist(),
    candidate_labels=classes_verbalized,
    multi_label=True,
    batch_size=batch_size
)

# Convertir la salida a formato legible
df["modelo_zeroshot"] = [
    str({label: float(score) for label, score in zip(r["labels"], r["scores"])})
    for r in results
]

# ------------------------------------------------------------
# RESULTADOS
# ------------------------------------------------------------
print("\n✅ Análisis completado!")
print(f"Total de títulos procesados: {len(df)}")
print(df[["texto_lower", "modelo_zeroshot"]])

Usando dispositivo: GPU
Analizando títulos en GPU con batching interno...

✅ Análisis completado!
Total de títulos procesados: 54191
                                             texto_lower  \
0      resolucion de de octubre de de la universidad ...   
1      resolucion de la universidad complutense de ma...   
2      resolucion de la universidad del pais vasco po...   
3      anuncio de la facultad de economicas y empresa...   
4      resolucion de la universidad de cordoba por la...   
...                                                  ...   
54186  resolucion de de marzo de de la universidad ca...   
54187  anuncio de la facultad de filosofia y ciencias...   
54188  anuncio de la facultad de enfermeria fisiotera...   
54189  anuncio de la facultad de medicina y odontolog...   
54190  anuncio de la facultad de medicina de la unive...   

                                         modelo_zeroshot  
0      {'empleo': 0.9924430847167969, 'tecnología': 0...  
1      {'tecnología': 0.9984

In [ ]:
# Guardado de los datos 
# df.to_csv("data_segundo_tercio_zeroshot.csv")